# 05. e + Ba: мишень, f(553.5), сечения против Fursa et al (1999), сечения из 6s5d

Линия **553.548 нм** (6s² ¹S₀ – 6s6p ¹P₁), A = 1.19·10⁸ с⁻¹, f = 1.64, ветвление ≈ 0.997.

Модель `ba1`: два валентных электрона над [Xe]; чётные 6s², 6s5d, 5d², 6p², 6s7s, 6s6d,
нечётные 6s6p, 5d6p, 6s7p, 6s4f; состояния до 4.1 эВ. Сильное КВ 6s6p ¹P – 5d6p ¹P
определяет f(553.5) — первая проверка мишени.

**Валидация** (самая сильная из трёх):
1. энергии и f(553.5) = 1.64 (A = 1.19·10⁸);
2. σ(¹P₁) прямое против CCC (Fursa, табл. II);
3. **σ_app = σ + каскады** против измерения Chen & Gallagher (±5 %) — проверяет и σ, и A;
4. остальные σ против рекомендованных Fursa (табл. III–VI, от 5 эВ).

Новое (нет нигде): σ из метастабилей 6s5d ³D, ¹D → 6s6p ¹P₁ и 5d6p; σ ниже 5 эВ.

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # эта папка: nbtools.py, run_model.py, models.py
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
SPECTRUM, SHEET = DATA / "A25_second.xlsx", "data2"   # измеренный спектр
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST (nist_*.csv) и список линий
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import crm, reference
import nbtools as nb
%matplotlib inline

In [ ]:
JMAX = 0            # 0 (проверка), затем 15-25
job = nb.Job(RUNS / "ba1", "ba1", stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=40, de=0.005, multipoles="E1,E2")
job.start()

In [ ]:
job.status()

In [ ]:
W = RUNS / "ba1"
print((W / "target_table.txt").read_text())

## 1. f и A резонансной линии и других линий Ba I против NIST

In [ ]:
nistA = [x for x in nb.nist_lines(NIST / "nist_BaI.csv") if x["A"]]
rows = nb.read_csv(W / "transitions_E1.csv")
for r in rows:
    wl = nb.fnum(r["wl_air_nm"])
    hit = [x for x in nistA if wl and abs(x["wl"] - wl) < 0.02]
    if hit:
        x = hit[0]; a = float(r["A_exp"])
        print(f"{x['wl']:9.3f} {r['upper_nist']:>18s} -> {r['lower_nist']:<16s} DBSR {a:.3e}  NIST {x['A']:.3e}  "
              f"ratio {a / x['A']:.2f}  gf {float(r['gf']):.3f}  vel/len {r['gauge_ratio']}")

## 2–3. σ(6s² → 6s6p ¹P₁): прямое против CCC, с каскадами против Chen & Gallagher

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz")
tg = db.Target.load(W / "target")
st = {s.name: s for s in tg.states}
lab = {n: st[n].nist_label for n in cs.names}
gs = cs.names[0]
p1 = [n for n in cs.names if lab[n] == "6s.6p 1P* J=1"][0]
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")
num = {n: st[n].nist_no for n in cs.names}
# каскад: σ_app(1P1) = σ(1P1) + Σ_u σ(u) · P(u -> ... -> 1P1); P через многократные ветвления
def reach(u, target, depth=4):
    if u == target: return 1.0
    if depth == 0 or u not in br: return 0.0
    return sum(b * reach(l, target, depth - 1) for l, b in br[u].items())
E = cs.incident_energy(gs)
direct = np.nan_to_num(cs.sigma(gs, p1))
casc = sum(np.nan_to_num(cs.sigma(gs, u)) * reach(num[u], num[p1]) for u in cs.names
           if st[u].exp_energy_cm > st[p1].exp_energy_cm)
F = nb.FURSA_1P1
plt.plot(E, direct / 1e-16, label="DBSR прямое")
plt.plot(E, (direct + casc) / 1e-16, label="DBSR + каскады")
plt.plot(F["E"], F["Q"], "s", ms=4, label="Fursa CCC (рек.), прямое")
plt.plot(F["E"], F["Q_app"], "o", ms=4, label="Chen & Gallagher ×1.03 (apparent)")
plt.xscale("log"); plt.xlim(2, 1000); plt.xlabel("E, эВ"); plt.ylabel("σ, 10⁻¹⁶ см²"); plt.legend(); plt.show()

## 4. Остальные переходы из основного против Fursa

In [ ]:
import re
def find_state(key):                       # "6s6p 3P1" -> state with NIST label "6s.6p 3P* J=1"
    conf, tj = key.split()
    c = ".".join(re.findall(r"\d[a-z](?:\d(?![a-z]))?", conf))
    want = {f"{c} {tj[:2]} J={tj[2:]}", f"{c} {tj[:2]}* J={tj[2:]}"}
    return [n for n in cs.names if lab[n] in want]
fig, axs = plt.subplots(3, 4, figsize=(15, 9))
for ax, key in zip(axs.flat, nb.FURSA_TABLES):
    Ef, sf = nb.fursa(key)
    ax.plot(Ef, sf / 1e-16, "s", ms=3, label="Fursa")
    for n in find_state(key)[:1]:
        ax.plot(cs.incident_energy(gs), cs.sigma(gs, n) / 1e-16, lw=0.8, label="DBSR")
    ax.set_xscale("log"); ax.set_title(key, fontsize=9)
axs.flat[0].legend(fontsize=7); fig.tight_layout(); plt.show()

## Новое: из метастабилей 6s5d в 6s6p ¹P₁ и константы скорости

In [ ]:
TE = np.array([0.5, 0.8, 1.1, 1.5, 2.0, 3.0])
meta = [n for n in cs.names if "6s.5d" in (lab[n] or "")]
for i in [gs] + meta:
    plt.plot(cs.incident_energy(i), cs.sigma(i, p1) / 1e-16, lw=0.8, label=f"{lab[i]} → ¹P₁")
    print(f"{lab[i]:>16s} -> 6s6p 1P1  k(Te) =", " ".join(f"{x:.2e}" for x in cs.rate(i, p1, TE * 11604.5)))
plt.xscale("log"); plt.xlabel("E, эВ"); plt.ylabel("σ, 10⁻¹⁶ см²"); plt.legend(fontsize=7); plt.show()

## CRM Ba I: насколько 553.5 отличается от корональной модели

Отношение I(553.5) из CRM к корональной оценке n_e·k(6s²→¹P₁) при разных ne и времени
пролёта τ. Если оно далеко от 1 — уравнение 2.1 диплома нужно заменить CRM.

In [ ]:
data = crm.CRMData.load(W / "crm_ba1.json")
g0, P1 = data.names[0], p1
kP = lambda te: float(np.exp(np.interp(np.log(te), np.log(data.Te), np.log(data.k[(0, data.index[P1])]))))
NE = np.logspace(9, 13, 30)
for tau in (2e-6, 2e-5):
    for te in (0.8, 1.1, 2.0):
        y = [crm.line_emission(data, crm.steady_state(data, ne, te, tau=tau), P1, g0) / (ne * kP(te) * 0.997)
             for ne in NE]
        plt.semilogx(NE, y, label=f"τ = {tau:g} с, Te = {te} эВ")
plt.xlabel("ne, см⁻³"); plt.ylabel("I(553.5): CRM / корональная"); plt.legend(fontsize=7); plt.show()